In [1]:
!pip install groq

In [2]:
from google.colab import userdata
from groq import Groq

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is empty. Add it in Colab Secrets.")
    client = Groq(api_key=GROQ_API_KEY)
    print("Groq client connected successfully!")
except Exception as error:
    raise RuntimeError(
        "Could not connect to Groq. Check that GROQ_API_KEY exists in Colab Secrets "
        "and that notebook access is enabled."
    ) from error

Groq client connected successfully!


In [3]:
SYSTEM_PROMPT = """
You are NutriBuddy, a friendly, context-aware nutrition information assistant.

Your responsibilities:
- Answer nutrition, food, hydration, meal-planning, and general healthy-eating questions.
- Use earlier messages in this conversation to understand follow-up questions.
- Use a user's name and details only if the user has shared them in this conversation.
- Do not invent personal details, measurements, allergies, diagnoses, or goals.
- Ask a short follow-up question when important information is missing.
- Explain nutrition concepts in simple, practical language.
- When giving meal ideas, consider dietary preferences, allergies, budget, and food availability mentioned by the user.
- Distinguish general educational information from individualized medical advice.
- Do not diagnose diseases, prescribe treatment, or recommend extreme diets.
- For pregnancy, eating disorders, chronic illness, medication-related nutrition, or other medical concerns, recommend guidance from a qualified healthcare professional.
- Never claim to be a registered dietitian or doctor.

Be supportive, clear, and conversational. Keep answers focused on the user's question.
"""

# This list is the agent's short-term conversational memory.
conversation_history = []

# Optional structured profile. Only add details the user has actually provided.
user_profile = {}

MODEL_NAME = "openai/gpt-oss-120b"

print("Agent instructions and session memory are ready.")

Agent instructions and session memory are ready.


In [4]:
def ask_nutrition_agent(user_message):
    global conversation_history

    user_message = user_message.strip()
    if not user_message:
        return "Please type a message."

    # Add the new message to the conversation memory.
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Include only the latest messages to keep the request manageable.
    recent_history = conversation_history[-20:]

    # Add profile information only when the user has supplied it.
    profile_context = ""
    if user_profile:
        profile_context = (
            "\\nUser-provided profile details (use only when relevant): "
            + str(user_profile)
        )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT + profile_context
        }
    ] + recent_history

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.5,
            max_completion_tokens=1000
        )

        answer = response.choices[0].message.content or "I couldn't generate a response."

        # Save the assistant's answer so the next question has full context.
        conversation_history.append({
            "role": "assistant",
            "content": answer
        })

        return answer

    except Exception as error:
        # Remove the unanswered user message so history remains consistent.
        conversation_history.pop()
        return f"Sorry, I couldn't reach the AI service. Error: {error}"


def reset_conversation():
    """Clear short-term chat history and the optional profile."""
    conversation_history.clear()
    user_profile.clear()
    print("Conversation and session profile cleared.")


def show_memory():
    """Display the messages currently stored in session memory."""
    if not conversation_history:
        print("No conversation history yet.")
        return

    for message in conversation_history:
        print(f"{message['role'].upper()}: {message['content']}\\n")

In [ ]:
print("I'm your Nutrition AI Agent!")
print("Ask a nutrition question. Type 'exit', 'reset', or 'memory' when needed.\\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("NutriBuddy: Goodbye! Take care of yourself. 👋")
        break

    elif user_input.lower() == "reset":
        reset_conversation()
        continue

    elif user_input.lower() == "memory":
        show_memory()
        continue

    reply = ask_nutrition_agent(user_input)
    print("\\nNutriBuddy:", reply, "\\n")

I'm your Nutrition AI Agent!
Ask a nutrition question. Type 'exit', 'reset', or 'memory' when needed.\n
